In [1]:
import pandas as pd
import requests
import logging
from uuid import uuid4
from simplejson import JSONDecodeError
import hmac
import hashlib
import base64
from datetime import datetime

In [2]:
log=logging.getLogger()
log.setLevel(logging.INFO)

In [3]:
class GLOBALS(object):
    enums=[]
    accounts=[]
    api_http_header={}
    tt_environment=''
    max_narrowing_retries=32,
    api_key = ''
    api_secret = ''

common = GLOBALS()
common.tt_environment='ext_prod_live'

In [4]:
def generate_hmac_header(api_key, api_secret, http_method, url_path):

    timestamp = datetime.now().strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3] + 'Z'
    message = f"{http_method.upper()} {url_path} {timestamp}"
    signature = hmac.new(
        api_secret.encode('utf-8'),
        message.encode('utf-8'),
        hashlib.sha256
    ).digest()
    signature_b64 = base64.b64encode(signature).decode('utf-8')
    auth_header = f'HMAC key="{api_key}", signature="{signature_b64}", timestamp="{timestamp}"'
    
    return {
        'Authorization': auth_header,
        'Content-Type': 'application/json',
        'x-api-key': api_key
    }

In [5]:
TT_URL_BASE='https://ttrestapi.trade.tt'
key = '55e5902f-199e-7904-c71f-a1a965cabdd4'
secret = '55e5902f-199e-7904-c71f-a1a965cabdd4:3bb82cd5-3527-1e91-8209-253d280a6d18'

TT_environments=(
    'ext_uat_cert',
    'ext_prod_live'   
)

In [6]:
def api_request(url,headers,data=None,http_method='get',request_timeout=False,userGivenparams={}):
    req_id='{}-{}--{}'.format('ttapi','SYMFI',uuid4()) #in here first is the any user given name for the app as to recognise for which app this call is being made and SYMFI is ours company name 
    print(req_id)
    params = {**userGivenparams,'requestId':req_id}
    # print(userGivenparams,'......................................................')
    response = \
        getattr(requests,http_method)(url=url,headers=headers,data=data,params=params)
    if request_timeout and response.status_code==408:
        raise AssertionError(
            "Erro on API request --> http code: {} message: {}".
            format(response.status_code, response.text)
        )
    try:
        response=response.json()

        if 'status' in response and response['status'] != 'Ok':
            print(response)
            error_message = \
                response['message'] if 'message' in response else response['status_message']
            raise AssertionError('Unable to retrieve data: {}'.format(error_message))
    except JSONDecodeError:
        raise AssertionError('Error decoding the response for {}'.format(url))
    
    return response
        


In [7]:
uuid4()

UUID('9dd71e36-2467-499b-a59d-2da6007c1fd3')

In [8]:
def retrieve_token(environment,key, secret, common):
    ttid_headers={
        'Content-Type': 'application/x-www-form-urlencoded',
        'content-type': 'application/json',
        'accept': 'application/json',
        'x-api-key': key
    }

    ttid_data={
        'grant_type': 'user_app',
        'app_key': secret
    }

    token_url = '{}/ttid/{}/token'.format(TT_URL_BASE,environment)

    try:
        login_info = api_request(url=token_url, headers=ttid_headers, data=ttid_data,  http_method='post')
        print(login_info)
    except AssertionError:
        raise

    token = '{} {}'.format(login_info['token_type'].capitalize(),login_info['access_token'])
    print(token)
    # token_expiry=time.time() + login_info['seconds_until_expiry']
    common.api_http_header={'x-api-key':key,'Authorization':token}
    
    

In [9]:
retrieve_token(common.tt_environment,key,secret,common)

ttapi-SYMFI--c7779979-d473-497b-8883-7c2b1153fa1a
{'status': 'Ok', 'access_token': 'eyJhbGciOiJSUzUxMiIsInR5cCI6IkpXVCIsIng1dCI6IjRqbnMxSzF3TVQ3M0V3cW1DVk9mZXJDTWlOZyJ9.eyJjbGllbnRfaWQiOiIxNjllMGY3MmI2NjA0N2NjODc0OGNkYTNjZWIyZGRlYSIsInN1YiI6IjE2OWUwZjcyYjY2MDQ3Y2M4NzQ4Y2RhM2NlYjJkZGVhIiwic2NvcGVzIjpbInJpc2thcGkvciIsInJpc2thcGkvdyIsIm1lc3NhZ2VjZW50ZXIvciIsImJpbGxpbmcvciIsImxlZGdlci9yIiwiYm9va2llL3IiLCJqdW5vd2ViL3IiLCJlZGdlL3ByIiwiZWRnZS9vciIsIm50dy94IiwicGRzL3IiLCJwZHMvdyIsImFuYWx5dGljcy9yIiwibWVzc2FnZWNlbnRlci93Iiwic2NvcmUvciIsInRyYWRlL3YiLCJhZGwvdiIsImFsZ290ZXN0aW5nL3YiLCJtb25pdG9yL3YiLCJzZXR1cC92IiwiaW5ib3gvdiIsInNjb3JlL3YiLCJzdGF0dXMvdiIsIm5leHRyYWRlciJdLCJuYmYiOjE3ODgyNzI0MDMsImV4cCI6MTc4ODMxNTY2MywiaXNzIjoiaHR0cHM6Ly9pZC50cmFkaW5ndGVjaG5vbG9naWVzLmNvbS8iLCJhdWQiOiJ0dC1leHQiLCJpZCI6NTg3MzgsImlkcHNuIjoidHQiLCJleHBpcmVzIjoxNzg4MzE1NjYzMDAwMDAwMDAwLCJyZWZyZXNoX2F0IjoxNzg4Mjc2MDYzLCJwZXJzb25faWQiOjQzMDAzLCJ0b2tlbl9pc3N1ZSI6MSwiYXBwX2lkIjoiMjAyMiIsImNsaWVudF9pcCI6IjM1Ljc1LjEzMC43MSIsImN

In [10]:
def create_new_user():
    """
    The /ttuser endpoint requires HMAC authentication, not Bearer token
    """
    url_path = '/ttuser/{}/user'.format(common.tt_environment)
    new_user_url = TT_URL_BASE + url_path
    
    # Generate HMAC headers instead of using Bearer token
    headers = generate_hmac_header(
        api_key=common.api_key,
        api_secret=common.api_secret,
        http_method='POST',
        url_path=url_path
    )
    
    try:
        response = api_request(new_user_url, headers, http_method='post')
        return response
    except AssertionError:
        raise


In [11]:
create_new_user()

ttapi-SYMFI--c52bb94d-81b2-4913-b84c-81c2d8721dd2


{'message': 'Forbidden'}

In [12]:
def get_account_position(accountId):
    position_url = '{}/ttmonitor/{}/position/{}'.format(TT_URL_BASE,common.tt_environment,accountId)
    try:
        positions = api_request(position_url,common.api_http_header)
        return positions
    except AssertionError:
        raise

In [ ]:
accounts = [1190712,1298430,1324415,1324439]

In [ ]:
positions=[]

In [ ]:
for acc in accounts:
    pos=get_account_position(acc)
    positions.extend(pos['positions'])

In [ ]:
positions

In [ ]:
instruments = []
for pos in positions:
    instruments.append(pos['instrumentId'])

In [ ]:
instruments=set(instruments)

In [13]:
def get_all_account(common):
    accounts_url=\
        '{}/ttaccount/{}/accounts/'.format(TT_URL_BASE,common.tt_environment)
    try:
        account_map=api_request(accounts_url,common.api_http_header)
        print(account_map['accounts'])
    except AssertionError:
        raise
    common.accounts=account_map['accounts']

In [14]:
get_all_account(common)

ttapi-SYMFI--8e4bb9c0-560b-4076-8f4a-589ae1a3e504
[{'accountType': 1, 'companyId': 14074, 'id': 1169688, 'name': 'LCE30553', 'parentAccountId': 1167892, 'parentId': 1167892, 'revision': 136205164}, {'accountType': 1, 'companyId': 14074, 'id': 1169689, 'name': 'LCE30555', 'parentAccountId': 1167892, 'parentId': 1167892, 'revision': 136205236}, {'accountType': 1, 'companyId': 14074, 'id': 1171766, 'name': '0SM03', 'parentAccountId': 1171743, 'parentId': 1171743, 'revision': 104137953}, {'accountType': 1, 'companyId': 14074, 'id': 1171769, 'name': '0SM02', 'parentAccountId': 1171743, 'parentId': 1171743, 'revision': 104137953}, {'accountType': 1, 'companyId': 14074, 'id': 1173381, 'name': 'LCE30565', 'parentAccountId': 1167892, 'parentId': 1167892, 'revision': 136205286}, {'accountType': 1, 'companyId': 14074, 'id': 1175886, 'name': '0SM04', 'parentAccountId': 1171743, 'parentId': 1171743, 'revision': 104137953}, {'accountType': 1, 'companyId': 14074, 'id': 1175888, 'name': '0SM05', 'pare

In [15]:
common.accounts

[{'accountType': 1,
  'companyId': 14074,
  'id': 1169688,
  'name': 'LCE30553',
  'parentAccountId': 1167892,
  'parentId': 1167892,
  'revision': 136205164},
 {'accountType': 1,
  'companyId': 14074,
  'id': 1169689,
  'name': 'LCE30555',
  'parentAccountId': 1167892,
  'parentId': 1167892,
  'revision': 136205236},
 {'accountType': 1,
  'companyId': 14074,
  'id': 1171766,
  'name': '0SM03',
  'parentAccountId': 1171743,
  'parentId': 1171743,
  'revision': 104137953},
 {'accountType': 1,
  'companyId': 14074,
  'id': 1171769,
  'name': '0SM02',
  'parentAccountId': 1171743,
  'parentId': 1171743,
  'revision': 104137953},
 {'accountType': 1,
  'companyId': 14074,
  'id': 1173381,
  'name': 'LCE30565',
  'parentAccountId': 1167892,
  'parentId': 1167892,
  'revision': 136205286},
 {'accountType': 1,
  'companyId': 14074,
  'id': 1175886,
  'name': '0SM04',
  'parentAccountId': 1171743,
  'parentId': 1171743,
  'revision': 104137953},
 {'accountType': 1,
  'companyId': 14074,
  'id':

In [16]:
accounts_list = ['EE839']

for account in common.accounts:
    if account['name'].lower() and any(substring.lower() in account['name'].lower() for substring in accounts_list):
        print(account)


In [ ]:
def get_fills(timeparams):
    fills_url = '{}/ttledger/{}/fills'.format(TT_URL_BASE,common.tt_environment)
    try:
        fills = api_request(fills_url,common.api_http_header,userGivenparams=timeparams)
    except Exception as e:
        raise
    return fills

In [ ]:
from datetime import datetime, timezone

def convert_time_to_ns(timestamp_str):
    if timestamp_str is None:
        return None
    
    # Parse the input timestamp (assumed to be in UTC)
    dt = datetime.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S.%f')
    
    # Make the datetime object timezone-aware (assuming UTC)
    dt = dt.replace(tzinfo=timezone.utc)

    # Convert to nanoseconds since epoch
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    ns_timestamp = int((dt - epoch).total_seconds() * 10**9)

    return ns_timestamp

def convert_ns_to_time(ns_timestamp):
    if ns_timestamp is None:
        return None
    
    # Convert nanoseconds to seconds
    seconds = ns_timestamp // 10**9  
    remainder_ns = ns_timestamp % 10**9  # Remaining nanoseconds

    # Convert to a timezone-aware datetime object (UTC)
    dt = datetime.fromtimestamp(seconds, tz=timezone.utc)

    # Format timestamp with milliseconds precision
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]



In [ ]:
minTime = convert_time_to_ns('2026-02-05 03:30:00.000')
maxTime = convert_time_to_ns('2026-05-06 03:30:00.000')
maxTime,minTime

In [ ]:
fills_array=[]
endTime = 0
currentMinTime = minTime
while currentMinTime < maxTime:
    print(currentMinTime)
    params = {'minTimestamp':str(currentMinTime),'maxTimestamp':str(maxTime), }
    fills = get_fills(params)['fills']
    fills_array.extend(fills)
    max = fills[0]['transactTime']
    for obj in fills:
        if obj['transactTime']>max:
            max = obj['transactTime']  
    if str(int(currentMinTime)-1) == obj['transactTime']:
        break;  
    currentMinTime = int(obj['transactTime'])+1
    

In [ ]:
dfnew=pd.DataFrame(fills_array)

In [ ]:
dfnew['time']=dfnew['transactTime'].astype(int).apply(convert_ns_to_time)

In [ ]:
pd.reset_option('display.max_rows')

In [ ]:
markets['id']=markets['id'].astype(int)

In [ ]:
dfnew=pd.merge(dfnew,markets,left_on='marketId',right_on='id',how='left')
dfnew

In [ ]:
dfnew.columns

In [ ]:
fills_array

In [ ]:
import json
with open('fill_array.json',"w") as file:
    json.dump(fills_array,file,indent=1)

In [ ]:
convert_ns_to_time(1741055358212000001)

In [ ]:
convert_ns_to_time(1741055399999000064)

In [ ]:
df = pd.DataFrame(fills_array)

In [ ]:
df['time']=df['transactTime'].astype(int).apply(convert_ns_to_time)

In [ ]:
df.columns

In [ ]:
df=df[['account','aggressorIndicator','deltaQty','instrumentId','lastQty','side','transactTime','time']]

In [ ]:
df[~df['aggressorIndicator'].isna()]

In [ ]:
df[df['aggressorIndicator'].isna()]

In [ ]:
df.to_csv('letsee2.csv',index=False)

In [ ]:
df['instrument']=df['instrumentId'].map(instrument_dict)

In [ ]:
df['instrument'].nunique()

In [ ]:
df.groupby(['account','instrument'])['lastQty'].sum().reset_index().to_csv('letsee.csv')

In [ ]:
xx=df.groupby('instrument')['lastQty'].sum().reset_index()

In [ ]:
xx[xx['instrument'].str.contains('er3',case=False)]

In [ ]:
df['instrumentId'].nunique()

In [ ]:
instruments=list(df['instrumentId'].unique())

In [ ]:
import time

In [ ]:
instrument_dict = {}
for inst in instruments:
    # print(inst)
    instrument_url='{}/ttpds/{}/instrument/{}'.format(TT_URL_BASE,common.tt_environment,str(inst))
    try:
        check=api_request(instrument_url,common.api_http_header)
    except Exception as e:
        print(e)
    # print(check)
    instrument_dict[inst]=check['instrument'][0]['alias']
    time.sleep(0.1)

In [ ]:
instrument_dict

In [ ]:
positions=pd.DataFrame(positions)

In [ ]:
positions['instrument']=positions['instrumentId'].map(instrument_dict)

In [ ]:
positions

In [ ]:
positions=positions[positions['netPosition']!=0]

In [ ]:
positions

In [ ]:
positions.to_excel(r'C:\mohit\positions.xlsx')

In [ ]:
with open("instruments.json","w") as file:
    json.dump(instrument_dict,file,indent=4)

In [ ]:
instrument_dict

In [ ]:
instrument_url='{}/ttpds/{}/instrument/{}'.format(TT_URL_BASE,common.tt_environment,str(15866520581074177814))
check=api_request(instrument_url,common.api_http_header)

In [ ]:
fills_array=get_fills()

In [ ]:
fills_array=fills_array['fills']

In [ ]:
maxYehhai = fills_array[0]['transactTime']
minYehhai = fills_array[0]['transactTime']
for obj in fills_array:
    if obj['transactTime'] > maxYehhai:
        maxYehhai = obj['transactTime']
    if obj['transactTime']<minYehhai:
        minYehhai = obj['transactTime']

In [ ]:
convert_ns_to_time(int(maxYehhai))

In [ ]:
convert_time_to_ns('2025-03-03 02:30:00.000')

In [ ]:
convert_ns_to_time(1740969000000000000)

In [ ]:
'2025-03-03 00:00:00.001 UTC'

In [ ]:
convert_ns_to_time(int(maxYehhai)),convert_ns_to_time(int(minYehhai))

In [ ]:
df=pd.DataFrame(get_fills()['fills'])

In [ ]:
pd.set_option('display.max_columns',None)

In [ ]:
df.sort_values('timeSentClient',inplace=True)

In [ ]:
df.dropna(subset='timeSentTT',inplace=True)

In [ ]:
df['timeSentTT']=df['timeSentTT'].astype(int)

In [ ]:
df['timeSentClient']=df['timeSentClient'].apply(convert_ns_to_ist)

In [ ]:
df['time']=df['timeSentTT'].apply(convert_ns_to_ist)

In [ ]:
df=df[['cumQty','deltaQty','exchLeavesQty','lastQty','tradeDate','fillsGroup','securityDesc','time']]

In [ ]:
df['tradeDATEsesrfsf']=df['tradeDate'].astype(int).apply(convert_ns_to_ist)

In [ ]:
def get_sod_for_all_instruments(accountId):
    fills_url=\
        '{}/ttmonitor/{}/sod/{}'.format(TT_URL_BASE,'ext_prod_sim',accountId)
    try:
        instrument_wise_sod=api_request(fills_url,common.api_http_header)['sod']
        # print(instrument_wise_sod)
        if len(instrument_wise_sod)!=0:
            for i in range(0,len(instrument_wise_sod)):
                instrument_url='{}/ttpds/{}/instrument/{}'.format(TT_URL_BASE,common.tt_environment,instrument_wise_sod[i]['instrumentId'])
                instrument_wise_sod[i]['instrumentAlias']=api_request(instrument_url,common.api_http_header)['instrument'][0]['alias']
                instrument_wise_sod[i]['month']=instrument_wise_sod[i]['instrumentId'][-5:]
    except AssertionError:
        raise
    return instrument_wise_sod

In [ ]:
get_sod_for_all_instruments('1283273')

In [ ]:
instrument_url='{}/ttpds/{}/instrument/{}'.format(TT_URL_BASE,common.tt_environment,'11152223431486213587')
api_request(instrument_url,common.api_http_header)

In [ ]:
markets_url='{}/ttpds/{}/markets'.format(TT_URL_BASE,common.tt_environment)
markets = api_request(markets_url,common.api_http_header)['markets']

In [ ]:
markets=pd.DataFrame(markets)

In [ ]:
with open("markets.json","w") as file:
    json.d

In [ ]:
instrument_url='{}/ttpds/{}/instrument/{}'.format(TT_URL_BASE,common.tt_environment,'17214838869451468739')
check=api_request(instrument_url,common.api_http_header)
check

In [ ]:
get_sod_for_all_instruments('1167892')

In [ ]:
instrument_wise_sod_Estr=get_sod_for_all_instruments('1167892')
instrument_wise_sod_Estr

In [ ]:
get_sod_for_all_instruments('1279482')

In [ ]:
instrument_wise_sod_I=get_sod_for_all_instruments('1279482')

In [ ]:
instrument_wise_sod_19=get_sod_for_all_instruments('1273152')

In [ ]:
instrument_wise_sod_I

In [ ]:
def get_sod_for_all_accounts():
    account_sods=[]
    try:
        if len(common.accounts)!=0:
            for i in range(len(common.accounts)):
            # for i in range(1):
                account_sod_url='{}/ttmonitor/{}/sod/{}'.format(TT_URL_BASE,common.tt_environment,common.accounts[i]['id'])
                account_sod=api_request(account_sod_url,common.api_http_header)['sod']
                if len(account_sod)!=0:
                    account_sods.append(account_sod)
                else:
                    print(common.accounts[i]['id'])
    except AssertionError:
        raise
    return account_sods

In [ ]:
account_sods=get_sod_for_all_accounts()

In [ ]:
import json

# Write JSON data to a file
with open('account_sods.json', 'w') as json_file:
    json.dump(account_sods, json_file, indent=4)

print("Data saved to data.json file")


In [ ]:
product_family_position_url='{}/ttpds/{}/productfamily/6146981086581524413'.format(TT_URL_BASE,common.tt_environment)
product_family_position=api_request(product_family_position_url,common.api_http_header)
product_family_position

In [ ]:
product_familly_details_url='{}/ttpds/{}/product/4228217215918507490'.format(TT_URL_BASE,common.tt_environment)
product_familly_details=api_request(product_familly_details_url,common.api_http_header)
product_familly_details